In [124]:
import pandas   as  pd
from openpyxl import load_workbook

In [125]:
id_realtion = pd.read_csv('/Users/leonardhaas/code/streamlit/data/raw_data/Umsteigeschluessel-KLDB2020-ISCO08.csv',sep=';',header=4)

In [126]:
id_realtion['Bezeichnungen der ISCO-08 (4-Steller)'] = id_realtion['ISCO-08\n(4-Steller)'].fillna('0').astype(int)

In [127]:
# Load the workbook
workbook = load_workbook('/Users/leonardhaas/code/streamlit/data/raw_data/Zensus22_Sonderauswertung_Haas.xlsx', read_only=True)

# Get all sheet names
sheet_names = workbook.sheetnames

berufs_gattungen_4 = pd.read_excel('/Users/leonardhaas/code/streamlit/data/raw_data/Zensus22_Sonderauswertung_Haas.xlsx', sheet_name=sheet_names[6], header=3)  # Row 2 as header (index 1)
berufs_gattungen_4

,ISCO-Code,Bezeichnung,Insgesamt,Baden-Württemberg,Bayern,Berlin,Brandenburg,Bremen,Hamburg,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Nordrhein-Westfalen,Rheinland-Pfalz,Saarland,Sachsen,Sachsen-Anhalt,Schleswig-Holstein,Thüringen
0,Insgesamt,Insgesamt,41043450,5667810,7024330,1772180,1208030,321090,948980,3048160,723350,3935110,8621290,2025940,470130,1837640,971950,1479240,988230
1,0110,Offiziere in regulären Streitkräften,22030,1200,2180,750,960,/,430,790,810,4690,4560,1740,380,1130,560,1100,620
2,0210,Unteroffiziere in regulären Streitkräften,22020,1690,2220,400,680,300,390,880,1470,4960,3620,1340,330,830,880,1340,700
3,0310,Angehörige der regulären Streitkräfte in sonst...,107160,7690,17460,2470,4320,800,1340,5640,4940,12340,20010,8300,1140,3710,4070,9510,3440
4,1111,Angehörige gesetzgebender Körperschaften,9740,1680,2250,290,/,/,/,700,240,830,1500,500,/,480,250,290,280
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416,9613,Straßenkehrer und verwandte Berufe,6760,520,990,750,250,/,/,550,/,700,1590,460,/,/,/,220,/
417,9621,"Boten, Paketauslieferer und Gepäckträger",157600,27650,27140,4030,3900,1370,1520,12650,2160,17170,31990,7860,1640,6750,3230,5580,2950
418,9622,Gelegenheitsarbeiter,690,/,/,/,/,/,/,/,/,/,270,/,/,/,/,/,/
419,9623,"Zählerableser, Automatenbefüller und -kassierer",9010,1200,1730,370,220,/,/,740,/,890,1890,500,/,260,/,370,230


In [128]:
def analyze_merge(left_df, right_df, left_on, right_on, how='left'):
    """
    Analyze merge performance between two dataframes.
    
    Parameters:
    -----------
    left_df : pandas.DataFrame
        The first (left) dataframe to merge
    right_df : pandas.DataFrame
        The second (right) dataframe to merge
    left_on : str
        Column name to merge on in the left dataframe
    right_on : str
        Column name to merge on in the right dataframe
    how : str, optional (default='left')
        Type of merge to perform. Options are 'left', 'right', 'inner', 'outer'
    
    Returns:
    --------
    tuple : (merged_dataframe, merge_analysis_report)
    """
    import pandas as pd
    
    # Perform merge with indicator
    merged_data = left_df.merge(
        right_df, 
        left_on=left_on, 
        right_on=right_on, 
        how=how,
        indicator=True
    )
    
    # Analyze merge results
    merge_counts = merged_data['_merge'].value_counts()
    
    # Get unmatched rows from left dataframe
    unmatched_rows = merged_data[merged_data['_merge'] == 'left_only']
    
    # Calculate match statistics
    total_rows = len(left_df)
    total_unique_keys = left_df[left_on].nunique()
    matched_unique_keys = merged_data[merged_data['_merge'] == 'both'][left_on].nunique()
    
    # Calculate match percentage based on unique keys
    match_percentage = (matched_unique_keys / total_unique_keys) * 100 if total_unique_keys > 0 else 0
    
    # Prepare detailed report
    report = {
        'total_rows_left': total_rows,
        'total_unique_keys_left': total_unique_keys,
        'matched_unique_keys': matched_unique_keys,
        'unmatched_unique_keys': total_unique_keys - matched_unique_keys,
        'merged_total_rows': len(merged_data),
        'match_percentage_unique_keys': match_percentage,
        'merge_type': how,
        'merge_counts': merge_counts.to_dict(),
        'unmatched_keys': unmatched_rows[left_on].unique().tolist()
    }
    
    # Print detailed report
    print("Merge Analysis Report:")
    print(f"Total rows in left dataset: {report['total_rows_left']}")
    print(f"Unique keys in left dataset: {report['total_unique_keys_left']}")
    print(f"Matched unique keys: {report['matched_unique_keys']}")
    print(f"Unmatched unique keys: {report['unmatched_unique_keys']}")
    print(f"Total rows in merged dataset: {report['merged_total_rows']}")
    print(f"Match Percentage (unique keys): {report['match_percentage_unique_keys']:.2f}%")
    print("\nMerge Counts:")
    for merge_type, count in report['merge_counts'].items():
        print(f"{merge_type}: {count}")
    
    # Remove the merge indicator column before returning
    merged_data = merged_data.drop(columns=['_merge'])
    
    return merged_data, report

In [129]:

merged_df, merge_report = analyze_merge(
     berufs_gattungen_4, 
     id_realtion, 
    left_on='ISCO-Code', 
     right_on='ISCO-08\n(4-Steller)'
)

Merge Analysis Report:
Total rows in left dataset: 421
Unique keys in left dataset: 421
Matched unique keys: 411
Unmatched unique keys: 10
Total rows in merged dataset: 1523
Match Percentage (unique keys): 97.62%

Merge Counts:
both: 1513
left_only: 10
right_only: 0


In [130]:
merged_df['KldB 2010\n(5-Steller)'] =merged_df['KldB 2010\n(5-Steller)'].fillna(0).astype(int)

In [131]:
verdienst_data_destatis= pd.read_csv('/Users/leonardhaas/code/streamlit/verdienst_destatis_kldb_5steller.csv')
#verdienst_data_destatis['code_kldb'] = verdienst_data_destatis['code_kldb'].astype(int)

In [132]:
final_merg,final_merge_report = analyze_merge(
    merged_df,
    verdienst_data_destatis,
    left_on='KldB 2010\n(5-Steller)',
    right_on='code_kldb'
)

Merge Analysis Report:
Total rows in left dataset: 1523
Unique keys in left dataset: 1297
Matched unique keys: 1296
Unmatched unique keys: 1
Total rows in merged dataset: 1523
Match Percentage (unique keys): 99.92%

Merge Counts:
both: 1513
left_only: 10
right_only: 0


In [133]:
final_merg.query("code_kldb==71104.0")

,ISCO-Code,Bezeichnung,Insgesamt,Baden-Württemberg,Bayern,Berlin,Brandenburg,Bremen,Hamburg,Hessen,...,Bezeichnungen der ISCO-08 (4-Steller),Unit Group (English),Umstieg eindeutig (1);\nnicht eindeutig (0),Schwerpunkt (1) und \nAnzahl der Alternativen,Unnamed: 8,Unnamed: 0,bezeichnung_kldb,median_brutto,average_brutto,code_kldb
7,1120,Geschäftsführer und Vorstände,487470,66190,95310,24110,15110,3320,15320,38700,...,1120.0,Managing directors and chief executives,1.0,1,NaN,840.0,71104 Geschäftsführer und Vorstände - Experte,7500.0,9132.0,71104.0


In [134]:
final_merg[['Bezeichnung','Insgesamt','code_kldb','bezeichnung_kldb','median_brutto','ISCO-Code']].sort_values(by=['median_brutto','ISCO-Code'],ascending=False)

,Bezeichnung,Insgesamt,code_kldb,bezeichnung_kldb,median_brutto,ISCO-Code
739,Flugzeugführer und verwandte Berufe,15520,52384.0,52384 Fahrzeugführer Flugverkehr (ssT)-Experte,25630.0,3153
737,Flugzeugführer und verwandte Berufe,15520,52314.0,"52314 Piloten,Verkehrsflugzeugführer - Experte",14120.0,3153
264,Zahnärzte,71880,81494.0,81494 Führung - Human- und Zahnmedizin,13095.0,2261
251,Fachärzte,255780,81494.0,81494 Führung - Human- und Zahnmedizin,13095.0,2212
44,Führungskräfte in der Erbringung von Dienstlei...,55410,81494.0,81494 Führung - Human- und Zahnmedizin,13095.0,1342
...,...,...,...,...,...,...
1507,Auf der Straße arbeitende Dienstleistungskräft...,930,NaN,NaN,NaN,9510
1451,Handwäscher und Handbügler,2160,NaN,NaN,NaN,9121
1248,Verspannungsmonteure und Seilspleißer,/,NaN,NaN,NaN,7215
764,Nicht akademische Fachkräfte in traditioneller...,1030,NaN,NaN,NaN,3230


In [135]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import plotly.io as pio

def generate_bubble_swarm_chart(df):
    # Validate input DataFrame
    required_columns = ['category', 'size_markers', 'value_y_axis']
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"DataFrame must contain columns: {required_columns}")
    
    # Ensure categories are strings
    df['category'] = df['category'].astype(str)
    
    # Get unique categories and assign colors
    categories = df['category'].unique()
    
    # Prepare color palette with fallback
    colors = [
        'rgba(31, 119, 180, 0.7)',   # Blue
        'rgba(255, 127, 14, 0.7)',   # Orange
        'rgba(44, 160, 44, 0.7)',]    # Green
     #   'rgba(214, 39, 40, 0.7)',    # Red
      #  'rgba(148, 103, 189, 0.7)'   # Purple
    #]
    
    # Extend colors if more categories than predefined colors
    if len(categories) > len(colors):
        # Generate additional colors if needed
        additional_colors = [f'rgba({np.random.randint(0,256)}, {np.random.randint(0,256)}, {np.random.randint(0,256)}, 0.7)' 
                              for _ in range(len(categories) - len(colors))]
        colors.extend(additional_colors)
    
    # Create traces
    traces = []
    for i, category in enumerate(categories):
        # Filter data for each category
        category_df = df[df['category'] == category]
        
        # Create randomized x-coordinates to spread out categories on the x-axis
        x = np.random.normal(loc=i*4, scale=0.3, size=len(category_df))

        y_jittered = category_df['value_y_axis'] + np.random.normal(loc=0, scale=0.1, size=len(category_df))
        
        # Create trace
        trace = go.Scatter(
            x=x,
            y=y_jittered,
            mode='markers',
            name=f'{category} (n={len(category_df)})',
            marker=dict(
                size=category_df['scale_markers'],
                color=colors[i],
                line=dict(width=1, color='rgba(0,0,0,0.5)'),
                opacity=0.7
            ),
            text=[f'<br>Subcategory: {subcategory}<br>Gehalt: {y:.2f}<br>Anzahl: {size:.2f}'
                  for y, size, subcategory in zip(category_df['value_y_axis'], 
                                                  category_df['size_markers'], 
                                                  category_df['subcategory'])],
            hoverinfo='text'
        )
        traces.append(trace)
    
    # Create layout
    layout = go.Layout(
        title='Klassenanalyse mit ISCO-Berufsgattung auf Basis von Zensusdaten',
        height=1500,
        width=800,
        yaxis=dict(
            title='Verdienst',
                    
                    autorange=True,  # Disable auto-ranging
                    rangemode='tozero'  # Ensure 0 is always visible
        ),
        hovermode='closest'
    )
    
    # Create and show figure
    fig = go.Figure(data=traces, layout=layout)
    pio.show(fig)
    
    # Optional: Save as HTML
    #pio.write_html(fig, file='bubble_swarm_chart.html')

# Debugging helper function
def print_dataframe_info(df):
    print("DataFrame Information:")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Number of rows: {len(df)}")
    print("\nCategory counts:")
    print(df['category'].value_counts())
    print("\nFirst few rows:")
    print(df.head())
    print("\nColumn dtypes:")
    print(df.dtypes)

In [136]:
income_plot = final_merg.iloc[1:]

#income_plot

In [137]:
income_plot['ISCO-Code']=income_plot['ISCO-Code'].astype(str)

/var/folders/05/nng50wx91lbfkkf4rcjhd68h0000gn/T/ipykernel_15246/4277457299.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [138]:
income_plot['isco_main_group']=income_plot['ISCO-Code'].str[0]

/var/folders/05/nng50wx91lbfkkf4rcjhd68h0000gn/T/ipykernel_15246/4007487838.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [139]:
income_plot.loc[
    (income_plot['isco_main_group'] == '2') |
    (income_plot['isco_main_group'] == '3') ,
    'fraktion'
] = 'Hochspezalisierte Beschäftigte'


/var/folders/05/nng50wx91lbfkkf4rcjhd68h0000gn/T/ipykernel_15246/1929540992.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [140]:
income_plot.loc[
    (income_plot['isco_main_group'] == '4') |
    (income_plot['isco_main_group'] == '5') |
    (income_plot['isco_main_group'] == '9') |
    (income_plot['isco_main_group'] == '3'),
    'fraktion'
] = 'Dienstleistungsarbeiter'


In [141]:
income_plot.loc[
    (income_plot['isco_main_group'] == '6') |
    (income_plot['isco_main_group'] == '7') |
    (income_plot['isco_main_group'] == '8'),
    'fraktion'
] = 'Industriearbeiter'


In [142]:
def pipline_data_bubble_chart(df):
    df = df[['fraktion','Insgesamt','Bezeichnung','median_brutto']]

    df.columns =['category', 'size_markers','subcategory','value_y_axis']
    df=df.replace('/','0')
    df['size_markers'] = df['size_markers'].astype(float)
    df.dropna(inplace=True)
    df['scale_markers'] = df['size_markers'] /35000
    return df


In [143]:
income_plot = pipline_data_bubble_chart(income_plot)

In [144]:
income_plot.drop_duplicates(inplace=True)

In [145]:
#income_plot.query('subcategory=="Bautechniker"')
income_plot.query("value_y_axis==0")

,category,size_markers,subcategory,value_y_axis,scale_markers
94,Hochspezalisierte Beschäftigte,27280.0,"Mathematiker, Versicherungsmathematiker und St...",0.0,0.779429
99,Hochspezalisierte Beschäftigte,41730.0,"Biologen, Botaniker, Zoologen und verwandte Be...",0.0,1.192286
105,Hochspezalisierte Beschäftigte,55410.0,"Agrar-, Forst- und Fischereiwissenschaftler un...",0.0,1.583143
151,Hochspezalisierte Beschäftigte,293490.0,Maschinenbauingenieure,0.0,8.385429
161,Hochspezalisierte Beschäftigte,43950.0,Chemieingenieure,0.0,1.255714
...,...,...,...,...,...
1348,Industriearbeiter,20950.0,"Näher, Sticker und verwandte Berufe",0.0,0.598571
1352,Industriearbeiter,960.0,"Pelzveredler, Gerber und Fellzurichter",0.0,0.027429
1362,Industriearbeiter,65880.0,"Handwerks- und verwandte Berufe, anderweitig n...",0.0,1.882286
1401,Industriearbeiter,71110.0,Bediener von Maschinen zur Herstellung von Nah...,0.0,2.031714


In [146]:
#TODO find a why to make a mean from the different values and drop 0 
#income_plot.groupby(['subcategory'])['value_y_axis'].sum()
income_plot['subcategory'].value_counts()

subcategory
Material- und ingenieurtechnische Fachkräfte, anderweitig nicht genannt    35
Bautechniker                                                               35
Produktionsleiter bei der Herstellung von Waren                            33
Maschinenbautechniker                                                      22
Ingenieure, anderweitig nicht genannt                                      22
                                                                           ..
Schmuckwarenhersteller und Edelmetallbearbeiter                             1
Fahrradmechaniker und verwandte Berufe                                      1
Finanzanalysten                                                             1
Flugmotorenmechaniker und -schlosser                                        1
Bürokräfte in der Lohnbuchhaltung                                           1
Name: count, Length: 382, dtype: int64

In [147]:
income_plot.sort_values(by='value_y_axis',ascending=False)

,category,size_markers,subcategory,value_y_axis,scale_markers
739,Dienstleistungsarbeiter,15520.0,Flugzeugführer und verwandte Berufe,25630.0,0.443429
737,Dienstleistungsarbeiter,15520.0,Flugzeugführer und verwandte Berufe,14120.0,0.443429
264,Hochspezalisierte Beschäftigte,71880.0,Zahnärzte,13095.0,2.053714
251,Hochspezalisierte Beschäftigte,255780.0,Fachärzte,13095.0,7.308000
741,Dienstleistungsarbeiter,5110.0,Flugverkehrslotsen,11716.0,0.146000
...,...,...,...,...,...
534,Dienstleistungsarbeiter,82210.0,Elektrotechniker,0.0,2.348857
1249,Industriearbeiter,14030.0,"Grobschmiede, Hammerschmiede und Schmiedepresser",0.0,0.400857
509,Dienstleistungsarbeiter,175530.0,Bautechniker,0.0,5.015143
472,Hochspezalisierte Beschäftigte,7950.0,"Bildende und darstellende Künstler, anderweiti...",0.0,0.227143


In [148]:
generate_bubble_swarm_chart(income_plot.query("value_y_axis !=0 and subcategory!='Flugzeugführer und verwandte Berufe'"))

/var/folders/05/nng50wx91lbfkkf4rcjhd68h0000gn/T/ipykernel_15246/2287430540.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

